# Year-over-Year Delinquency Driver Analysis

This portfolio notebook demonstrates an end-to-end comparison of two origination cohorts using **synthetic consumer-credit data**.

It covers feature engineering, regularized logistic regression, cross-validation, holdout evaluation, threshold analysis, coefficient and permutation importance, and a two-part composition-versus-within-segment decomposition.

> This is an observational driver analysis. The results describe adjusted associations and decomposition effects; they do not establish causality.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option("display.max_columns", 100)

## 1. Synthetic cohort generation

In [ ]:
def make_synthetic_cohort(year: int, n: int, rng: np.random.Generator) -> pd.DataFrame:
    is_2025 = int(year == 2025)

    ratings = np.array(["A", "B", "C", "D", "E", "X"])
    rating_probs = (
        np.array([0.18, 0.24, 0.24, 0.17, 0.11, 0.06])
        if year == 2024
        else np.array([0.15, 0.21, 0.24, 0.19, 0.13, 0.08])
    )

    app_types = np.array(["STANDARD", "NEW", "REPEAT", "PROMO"])
    app_probs = (
        np.array([0.44, 0.24, 0.24, 0.08])
        if year == 2024
        else np.array([0.39, 0.30, 0.22, 0.09])
    )

    app_statuses = np.array(["OPEN", "RE-OPEN", "RENEWAL"])
    sites = np.array([f"{i:03d}" for i in range(101, 131)])
    district_map = {site: f"DISTRICT_{1 + i // 5}" for i, site in enumerate(sites)}

    rating = rng.choice(ratings, n, p=rating_probs)
    app_type = rng.choice(app_types, n, p=app_probs)
    app_stat = rng.choice(app_statuses, n, p=[0.72, 0.12, 0.16])
    site_id = rng.choice(sites, n)
    district = pd.Series(site_id).map(district_map).to_numpy()

    score_mean = {"A": 735, "B": 690, "C": 650, "D": 610, "E": 565, "X": 510}
    trans_union_score = np.array(
        [rng.normal(score_mean[r], 28) for r in rating]
    ).clip(350, 820).round()

    purchase = rng.lognormal(7.0, 0.65, n).clip(100, 12000)
    dp_pct = np.clip(
        rng.beta(2.2, 8.0, n)
        + np.where(rating == "A", 0.05, 0)
        - np.where(np.isin(rating, ["E", "X"]), 0.03, 0),
        0,
        0.8,
    )
    down_payment = purchase * dp_pct
    loan = np.maximum(purchase - down_payment, 0)

    rating_effect = pd.Series(rating).map({
        "A": -1.05, "B": -0.60, "C": -0.20,
        "D": 0.25, "E": 0.65, "X": 1.00,
    }).to_numpy()
    app_effect = pd.Series(app_type).map({
        "STANDARD": 0.00, "NEW": 0.38, "REPEAT": -0.22, "PROMO": 0.18,
    }).to_numpy()
    status_effect = pd.Series(app_stat).map({
        "OPEN": 0.00, "RE-OPEN": -0.32, "RENEWAL": -0.12,
    }).to_numpy()
    district_effect = pd.Series(district).map({
        f"DISTRICT_{i}": effect
        for i, effect in enumerate([-0.12, 0.05, 0.18, -0.04, 0.10, -0.08], start=1)
    }).to_numpy()

    logit = (
        -1.20
        + rating_effect
        + app_effect
        + status_effect
        + district_effect
        - 1.10 * dp_pct
        + 0.16 * np.log1p(loan)
        + 0.16 * is_2025
        + 0.10 * is_2025 * np.isin(rating, ["D", "E", "X"])
    )
    probability = 1 / (1 + np.exp(-logit))
    is_dlq = rng.binomial(1, probability)

    dlq_code = np.where(
        is_dlq == 1,
        rng.choice(["2", "3", "4", "5"], n, p=[0.55, 0.25, 0.14, 0.06]),
        rng.choice(["0", "1"], n, p=[0.88, 0.12]),
    )

    return pd.DataFrame({
        "cohort_year": year,
        "site_id": site_id,
        "district": district,
        "rating_code_calc": rating,
        "trans_union_score": trans_union_score,
        "app_stat": app_stat,
        "app_type": app_type,
        "sum_purch_amt": purchase.round(2),
        "sum_dp_amt": down_payment.round(2),
        "sum_loan_amt": loan.round(2),
        "dlq_code": dlq_code,
        "is_dlq": is_dlq,
    })


df24 = make_synthetic_cohort(2024, 8_000, rng)
df25 = make_synthetic_cohort(2025, 8_000, rng)
both = pd.concat([df24, df25], ignore_index=True)
both.head()

## 2. Feature engineering

In [ ]:
PURCH_BINS = [-np.inf, 500, 1000, 1500, 2000, 3000, 5000, np.inf]
PURCH_LABELS = ["<=500", "501-1K", "1K-1.5K", "1.5K-2K", "2K-3K", "3K-5K", "5K+"]

SCORE_BINS = [-np.inf, 499, 549, 599, 649, 699, 749, np.inf]
SCORE_LABELS = ["<500", "500-549", "550-599", "600-649", "650-699", "700-749", "750+"]

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result["purch_bucket"] = pd.cut(
        result["sum_purch_amt"], bins=PURCH_BINS, labels=PURCH_LABELS
    ).astype(str)
    result["trans_score_bucket"] = pd.cut(
        result["trans_union_score"], bins=SCORE_BINS, labels=SCORE_LABELS
    ).astype(str)
    result["dp_pct"] = (
        result["sum_dp_amt"].abs() / result["sum_purch_amt"].abs()
    ).clip(0, 1).fillna(0)
    result["log_loan"] = np.log1p(result["sum_loan_amt"].clip(lower=0))
    result["log_sale"] = np.log1p(result["sum_purch_amt"].clip(lower=0))
    result["year"] = (result["cohort_year"] == 2025).astype(int)
    return result

both = engineer_features(both)
df24 = both.query("cohort_year == 2024").copy()
df25 = both.query("cohort_year == 2025").copy()

both.groupby("cohort_year").agg(
    accounts=("is_dlq", "size"),
    delinquency_rate=("is_dlq", "mean"),
)

## 3. Logistic regression and validation

In [ ]:
CAT_COLS = [
    "rating_code_calc",
    "trans_score_bucket",
    "app_stat",
    "app_type",
    "district",
    "purch_bucket",
]
NUM_COLS = ["dp_pct", "log_loan", "log_sale", "year"]
FEATURE_COLS = CAT_COLS + NUM_COLS

X = both[FEATURE_COLS]
y = both["is_dlq"]

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
        CAT_COLS,
    ),
    ("numeric", StandardScaler(), NUM_COLS),
], verbose_feature_names_out=False)

model = Pipeline([
    ("preprocess", preprocessor),
    ("logistic", LogisticRegression(
        max_iter=2000,
        C=1.0,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
    },
    n_jobs=-1,
)

pd.DataFrame({
    name.replace("test_", ""): {
        "mean": values.mean(),
        "std": values.std(),
    }
    for name, values in cv_results.items()
    if name.startswith("test_")
}).T

In [ ]:
test_roc_auc = roc_auc_score(y_test, test_probability)
test_pr_auc = average_precision_score(y_test, test_probability)

null_probability = np.full(len(y_test), y_test.mean())
ll_model = -log_loss(y_test, test_probability, normalize=False)
ll_null = -log_loss(y_test, null_probability, normalize=False)
mcfadden_r2 = 1 - (ll_model / ll_null)

pd.DataFrame({
    "metric": ["Holdout ROC-AUC", "Holdout PR-AUC", "Holdout McFadden pseudo R²"],
    "value": [test_roc_auc, test_pr_auc, mcfadden_r2],
})

## 4. Holdout ROC, precision-recall, and thresholds

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_probability)
precision, recall, _ = precision_recall_curve(y_test, test_probability)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {test_roc_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Random")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Holdout ROC curve")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, label=f"PR-AUC = {test_pr_auc:.3f}")
ax.axhline(y_test.mean(), linestyle="--", label=f"Baseline = {y_test.mean():.1%}")
ax.set(xlabel="Recall", ylabel="Precision", title="Holdout precision-recall curve")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
rows = []
for threshold in np.arange(0.20, 0.66, 0.05):
    predicted = (test_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    rows.append({
        "threshold": threshold,
        "precision": p,
        "recall": r,
        "f1": f1,
        "flagged_rate": predicted.mean(),
    })

threshold_df = pd.DataFrame(rows)
threshold_df.round(3)

In [ ]:
print(classification_report(y_test, test_prediction, target_names=["Current", "DLQ"]))

pd.DataFrame(
    confusion_matrix(y_test, test_prediction),
    index=["Actual current", "Actual DLQ"],
    columns=["Predicted current", "Predicted DLQ"],
)

## 5. Coefficient-based importance

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefficients = model.named_steps["logistic"].coef_[0]

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "odds_ratio_per_scaled_unit": np.exp(coefficients),
        "absolute_coefficient": np.abs(coefficients),
    })
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

coef_df.head(20)

In [ ]:
top = coef_df.head(20).sort_values("coefficient")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top["feature"], top["coefficient"])
ax.axvline(0, linewidth=0.8)
ax.set(xlabel="Logistic coefficient", title="Top standardized coefficients")
ax.grid(axis="x", alpha=0.3)
plt.show()

Numeric variables are standardized. Categorical coefficients are relative to the omitted reference category. Coefficients represent adjusted associations, not causal effects.

## 6. Holdout permutation importance

In [ ]:
permutation = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "feature_group": FEATURE_COLS,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

perm_df

In [ ]:
plot_perm = perm_df.sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_perm["feature_group"], plot_perm["importance_mean"])
ax.set(xlabel="Mean holdout ROC-AUC decrease", title="Permutation importance")
ax.grid(axis="x", alpha=0.3)
plt.show()

## 7. Composition and within-segment rate decomposition

In [ ]:
def decompose_change(prior, current, segment_col, outcome_col="is_dlq"):
    prior_rate = prior.groupby(segment_col)[outcome_col].mean()
    current_rate = current.groupby(segment_col)[outcome_col].mean()
    prior_weight = prior[segment_col].value_counts(normalize=True)
    current_weight = current[segment_col].value_counts(normalize=True)

    segments = sorted(set(prior_rate.index) | set(current_rate.index))
    detail = pd.DataFrame({"segment": segments})
    detail["prior_rate"] = detail["segment"].map(prior_rate)
    detail["current_rate"] = detail["segment"].map(current_rate)
    detail["prior_weight"] = detail["segment"].map(prior_weight).fillna(0)
    detail["current_weight"] = detail["segment"].map(current_weight).fillna(0)
    detail = detail.dropna(subset=["prior_rate", "current_rate"])

    detail["composition_contribution"] = (
        detail["current_weight"] - detail["prior_weight"]
    ) * detail["prior_rate"]
    detail["within_segment_contribution"] = (
        detail["current_rate"] - detail["prior_rate"]
    ) * detail["current_weight"]
    detail["total_contribution"] = (
        detail["composition_contribution"] + detail["within_segment_contribution"]
    )

    prior_total = (detail["prior_rate"] * detail["prior_weight"]).sum()
    current_total = (detail["current_rate"] * detail["current_weight"]).sum()
    counterfactual = (detail["prior_rate"] * detail["current_weight"]).sum()

    composition = counterfactual - prior_total
    within_segment = current_total - counterfactual
    total = current_total - prior_total

    return detail.sort_values("total_contribution", ascending=False), composition, within_segment, total


decomposition_rows = []
decomposition_details = {}

for column in [
    "rating_code_calc",
    "app_type",
    "trans_score_bucket",
    "app_stat",
    "district",
    "purch_bucket",
]:
    detail, composition, within_segment, total = decompose_change(df24, df25, column)
    decomposition_details[column] = detail
    decomposition_rows.append({
        "feature": column,
        "total_change_pp": 100 * total,
        "composition_effect_pp": 100 * composition,
        "within_segment_effect_pp": 100 * within_segment,
        "composition_share": composition / total if total else np.nan,
        "within_segment_share": within_segment / total if total else np.nan,
    })

decomposition_df = pd.DataFrame(decomposition_rows)
decomposition_df.round(3)

In [ ]:
plot_decomp = decomposition_df.set_index("feature")[
    ["composition_effect_pp", "within_segment_effect_pp"]
]
fig, ax = plt.subplots(figsize=(9, 5))
plot_decomp.plot(kind="bar", ax=ax)
ax.axhline(0, linewidth=0.8)
ax.set(
    xlabel="Feature",
    ylabel="Contribution to rate change (percentage points)",
    title="Composition vs. within-segment decomposition",
)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Each feature is decomposed separately. Percentages from different feature decompositions should not be added together.

## 8. Adjusted cohort-year effect

In [ ]:
year_index = list(feature_names).index("year")
year_coefficient = coefficients[year_index]
year_odds_ratio = np.exp(year_coefficient)

pd.DataFrame({
    "measure": [
        "Standardized cohort-year coefficient",
        "Adjusted cohort-year odds ratio",
    ],
    "value": [year_coefficient, year_odds_ratio],
})

The cohort-year coefficient is an adjusted association after controlling for observed features. It is not a causal treatment effect or a complete estimate of “2025 conditions.”

## 9. Reproducible summary

In [ ]:
summary_output = pd.DataFrame({
    "metric": [
        "2024 delinquency rate",
        "2025 delinquency rate",
        "Observed YoY change (pp)",
        "Holdout ROC-AUC",
        "Holdout PR-AUC",
        "Holdout McFadden pseudo R²",
        "Adjusted cohort-year odds ratio",
    ],
    "value": [
        df24["is_dlq"].mean(),
        df25["is_dlq"].mean(),
        100 * (df25["is_dlq"].mean() - df24["is_dlq"].mean()),
        test_roc_auc,
        test_pr_auc,
        mcfadden_r2,
        year_odds_ratio,
    ],
})
summary_output

## Limitations

- The data are synthetic and demonstrate workflow rather than a real portfolio.
- Logistic regression estimates conditional associations and does not establish causality.
- Decomposition results depend on the selected segment and reference period.
- Unobserved variables may explain part of the cohort difference.
- Threshold selection should reflect operational false-positive and false-negative costs.